# Deep Q Learning for Lunar Landing

## Part 1- Installin Required Packages and importing the Libraries

### Installing gymnasium

In [36]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!apt-get install -y swig
!pip install gymnasium[box2d]


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


### Importing libraries

In [37]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.autograd as autograd
from torch.autograd import Variable
from collections import deque, namedtuple

## Part 2 - Building the AI

Creating the architecture of the Neural Network.

In [70]:
class Network(nn.Module):
  def __init__(self,state_size,action_size,seed=42)-> None:
    super(Network,self).__init__( )
    self.seed=torch.manual_seed(seed)
    self.fc1=nn.Linear(state_size,64)
    self.fc2=nn.Linear(64,64)
    self.fc3=nn.Linear(64,action_size)
  def forward(self,state):
    x=F.relu(self.fc1(state))
    x=F.relu(self.fc2(x))
    x=self.fc3(x)
    return x


## Part 3 - Training the AI

### Setting up the Environment

In [71]:
import gymnasium as gym
env=gym.make('LunarLander-v3')
state_shape=env.observation_space.shape
state_size=env.observation_space.shape[0]
number_actions=env.action_space.n
print("State size:",state_size)
print("State shape:",state_shape)
print("Number of actions:",number_actions)

State size: 8
State shape: (8,)
Number of actions: 4


### Initializing the hyperparameters

In [72]:
learning_rate=5e-4
minibatch_size=100
discount_factor=0.99
replay_buffer_size=int(1e5)
interpolation_parameter=1e-2

### Implementing Experience Replay

In [96]:
class ReplayMemory(object):
    def __init__(self, capacity, device=None):
        self.device = device if device is not None else torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.capacity = capacity
        self.memory = []

    def push(self, event):
        self.memory.append(event)
        if len(self.memory) > self.capacity:
            del self.memory[0]

    def sample(self, batch_size):
        batch_size = min(batch_size, len(self.memory))
        experiences = random.sample(self.memory, k=batch_size)
        shaped_rewards = []
        for e in experiences:
            state, action, reward, next_state, done = e
            x_pos = next_state[0]
            x_vel = next_state[2]
            reward_shaped = reward - 0.2 * abs(x_pos)
            shaped_rewards.append(reward_shaped)
        states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float().to(self.device)
        actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long().to(self.device)
        rewards = torch.from_numpy(np.vstack(shaped_rewards)).float().to(self.device)
        next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float().to(self.device)
        dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.float32)).float().to(self.device)
        return states, next_states, actions, rewards, dones

    def __len__(self):
        return len(self.memory)

### Implementing the DQN class

In [105]:
class Agent():
  def __init__(self,state_size,action_size):
    self.device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    self.state_size=state_size
    self.action_size=action_size
    self.local_qnetwork=Network(state_size,action_size).to(self.device)
    self.target_qnetwork=Network(state_size,action_size).to(self.device)
    self.optimizer=optim.Adam(self.local_qnetwork.parameters(),lr=learning_rate)
    self.memory=ReplayMemory(replay_buffer_size)
    self.t_step=0
  def step(self, state, action, reward, next_state, done):
    x_pos = next_state[0]
    x_vel = next_state[2]
    reward -= 0.1 * abs(x_pos)
    reward -= 0.05 * abs(x_vel)
    self.memory.push((state, action, reward, next_state, done))
    self.t_step = (self.t_step + 1) % 4
    if self.t_step == 0:
        if len(self.memory.memory) > minibatch_size:
            experiences = self.memory.sample(minibatch_size)
            self.learn(experiences, discount_factor)
  def act(self,state,epsilon=0.):
    state=torch.from_numpy(state).float().unsqueeze(0).to(self.device)
    self.local_qnetwork.eval()
    with torch.no_grad():
      action_values=self.local_qnetwork(state)
    self.local_qnetwork.train()
    if random.random()>epsilon:
      return np.argmax(action_values.cpu().data.numpy())
    else:
      return random.choice(np.arange(self.action_size))

  def learn(self,experiences,discount_factor):
    states,next_states,actions,rewards,dones=experiences
    next_q_targets=self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)
    q_targets=rewards+(discount_factor*next_q_targets*(1-dones))
    q_expected=self.local_qnetwork(states).gather(1,actions)
    loss=F.mse_loss(q_expected,q_targets)
    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()
    self.soft_update(self.local_qnetwork,self.target_qnetwork,interpolation_parameter)

  def soft_update(self,local_model,target_model,interpolation_parameter):
    for target_param,local_param in zip(target_model.parameters(),local_model.parameters()):
      target_param.data.copy_(interpolation_parameter*local_param+(1.0-interpolation_parameter)*target_param.data)


### Initializing the DQN agent

In [106]:
agent=Agent(state_size,number_actions)

### Training DQN agent

In [107]:
number_of_episodes=2000
maximum_number_timesteps_per_episode=1000
epsilon_starting_value=1.0
epsilon_ending_value=0.01
epsilon_decay_value=0.995
epsilon=epsilon_starting_value
scores_on_100_episodes=deque(maxlen=100)

In [108]:
for episode in range(1,number_of_episodes+1):
  state,_=env.reset()
  score=0
  for t in range(maximum_number_timesteps_per_episode):
    action=agent.act(state,epsilon)
    next_state,reward,done,_,_=env.step(action)
    agent.step(state,action,reward,next_state,done)
    state=next_state
    score+=reward
    if done:
      break
  scores_on_100_episodes.append(score)
  epsilon=max(epsilon_ending_value,epsilon_decay_value*epsilon)
  print("\rEpisode {}\tAverage Score:{:2f}".format(episode,np.mean(scores_on_100_episodes)),end="")
  if episode % 100==0:
    print("\rEpisode {}\tAverage Score:{:2f}".format(episode,np.mean(scores_on_100_episodes)))
  if np.mean(scores_on_100_episodes)>=200.0:
    print("\nEnvironment solved in {:d} episodes!\tAverage Score:{:.2f}".format(episode-100,np.mean(scores_on_100_episodes)))
    torch.save(agent.local_qnetwork.state_dict(),'checkpoint.pth')
    break





Episode 100	Average Score:-155.370694
Episode 200	Average Score:-68.426246
Episode 300	Average Score:-0.205189
Episode 400	Average Score:137.344186
Episode 440	Average Score:200.636969
Environment solved in 340 episodes!	Average Score:200.64


## Part 4 - Visualization

In [111]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display
from gym.wrappers.monitoring.video_recorder import VideoRecorder

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.act(state)
        state, reward, done, _, _ = env.step(action.item())
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'LunarLander-v3')

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()